In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import numpy as np
import math
from datetime import datetime, timezone, timedelta
import os
import cv2
import glob
import pandas as pd
import glob
import re
from tqdm import tqdm
import matplotlib.colors as mcolors
from collections import defaultdict
import folium
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from moviepy.video.VideoClip import TextClip, ImageClip
from moviepy.video.compositing.CompositeVideoClip import CompositeVideoClip

BASE_PATH = "/content/drive/MyDrive/TOMTOM DATA"
DATASET_PATH = os.path.join(BASE_PATH, "Dataset")
onset_congestion_map_dir = os.path.join(BASE_PATH, "On-set Congestion Map")
onset_overlay_map_dir = os.path.join(onset_congestion_map_dir, "overlay_map")
traffic_timelapse_dir = os.path.join(BASE_PATH, "traffic_timelapse")

START_DATE="2026-04-01"
END_DATE="2026-04-15"

os.makedirs(onset_congestion_map_dir, exist_ok=True)
os.makedirs(onset_overlay_map_dir, exist_ok=True)
os.makedirs(traffic_timelapse_dir, exist_ok=True)


THETA = 0.9   # congestion threshold (yellow or worse)

MAIN_COLORS = np.array([
    (119, 119, 119),  # #777777 Grey
    (255, 35, 35),    # #FF2323 Red
    (255, 255, 55),   # #FFFF37 Yellow
    (43, 200, 43)     # #2BC82B Green

], dtype=np.int32)

COLOR_WEIGHTS = np.array([0.005, # Grey (Complete Congestion)
                          0.405, # Red (Congestion)
                          0.9,   # Yellow (Light Congestion)
                          1])    # Green (No Congestion)

CITY_CORD_DICT = {
    "Riyadh": {"lat": 24.7136, "lon": 46.6753},
    "Jeddah": {"lat": 21.5294, "lon": 39.1611},
    "Dammam": {"lat": 26.4241, "lon": 50.0905},
    "Al khobar": {"lat": 26.2199, "lon": 50.1932},
    "Dhahran": {"lat": 26.2381, "lon": 50.0430},
    "Al Qatif": {"lat": 26.5781, "lon": 49.9985},
}

In [ ]:
def parse_filename(path):
    name = os.path.basename(path).replace(".png", "")
    parts = name.split("_")

    raw_ts = parts[7]

    # Fix time separators using regex
    fixed_ts = re.sub(
        r"T(\d{2})-(\d{2})-(\d{2})",
        r"T\1:\2:\3",
        raw_ts
    )

    fixed_ts = re.sub(
        r"\+(\d{2})-(\d{2})",
        r"+\1:\2",
        fixed_ts
    )

    metadata = {
        "city": parts[1],
        "index": int(parts[2]),
        "latitude": float(parts[3]),
        "longitude": float(parts[4]),
        "zoom": int(parts[5][1:]),     # remove 'z'
        "radius": int(parts[6][1:]),   # remove 'r'
        "timestamp": datetime.fromisoformat(fixed_ts)
    }

    return metadata

def to_datetime_safe(x):
    DEFAULT_TZ = timezone(timedelta(hours=3))  # +03:00

    if x is None:
        return None

    if isinstance(x, datetime):
        dt = x
    else:
        dt = datetime.fromisoformat(x)

    # If naive → attach timezone
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=DEFAULT_TZ)

    return dt


def load_city_images_paths(base_dir, start_date=None, end_date=None):
    city_dict = {}
    start_date = to_datetime_safe(start_date)
    end_date = f"{end_date}T23:59" if end_date else None
    end_date = to_datetime_safe(end_date)

    for city_name in os.listdir(base_dir):
        city_path = os.path.join(base_dir, city_name)

        if not os.path.isdir(city_path):
            continue

        city_map_path = os.path.join(city_path, "stitched")
        if not os.path.isdir(city_map_path):
            continue

        image_paths = glob.glob(os.path.join(city_map_path, "*.png"))

        filtered_paths = []

        for path in image_paths:
            meta = parse_filename(path)
            ts = meta["timestamp"]

            if start_date and ts < start_date:
                continue
            if end_date and ts > end_date:
                continue

            filtered_paths.append(path)

        city_dict[city_name] = filtered_paths

    return city_dict

city_dict = load_city_images_paths(DATASET_PATH, start_date=START_DATE, end_date=END_DATE)

In [3]:
def get_img(img_path):
    "Rteturn color calibrated image as ndarry"
    try:
        img = Image.open(img_path)
        return calibrate_colors(img)
    except FileNotFoundError:
        print(f"Error: The file '{img_path}' was not found. Please ensure you have uploaded the image to your Colab environment and that the filename is correct.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return None

def calibrate_colors(img):
    "Make sure that colors follow restricted color code"
    img_rgb = img.convert("RGB")
    img_array = np.asarray(img_rgb, dtype=np.uint8)

    # Reshape image to (num_pixels, 3)
    h, w, _ = img_array.shape
    flat_pixels = img_array.reshape(-1, 3).astype(np.int16)

    # Compute squared distances to each main color
    min_dist = np.full(flat_pixels.shape[0], np.inf)
    min_idx = np.zeros(flat_pixels.shape[0], dtype=np.int32)

    for i, color in enumerate(MAIN_COLORS):
        diff = flat_pixels - color  # (N, 3)
        dist = np.sum(diff * diff, axis=1)  # (N,)

        mask = dist < min_dist
        min_dist[mask] = dist[mask]
        min_idx[mask] = i

    # Apply threshold
    threshold_sq = 150 ** 2
    mask = min_dist <= threshold_sq

    # Create output array (default black)
    new_pixels = np.zeros_like(flat_pixels)

    # Replace only pixels within threshold
    new_pixels[mask] = MAIN_COLORS[min_idx[mask]]

    # Reshape back to image
    return new_pixels.reshape(h, w, 3).astype(np.uint8)

def img_to_weight_map(path):

    img = get_img(path)

    h, w, _ = img.shape
    weight_map = np.zeros((h, w))
    mask = None
    for color, weight in zip(MAIN_COLORS, COLOR_WEIGHTS):
        mask = np.all(img == color, axis=-1)
        weight_map[mask] = weight

    return weight_map

def average_img_weight_map(image_paths):
    """
    image_paths: list of image file paths taken at same time (different days)
    returns: average weight matrix (h, w)
    """

    running_sum = None
    count = 0
    for path in tqdm(image_paths, "Get Weight Map Average"):
        wm = img_to_weight_map(path)
        if running_sum is None:
            running_sum = np.zeros_like(wm, dtype=np.float32)
        running_sum += wm
        count += 1
        del wm  # important in Jupyter

    if count == 0:
        return None

    return running_sum / count


def prepare_onset_map_plt(relative_onset, road_mask, cmap, norm):

    h, w = relative_onset.shape
    output = np.zeros((h, w, 3), dtype=np.uint8)

    # Road but never congested → Grey
    never_congested = road_mask & np.isnan(relative_onset)
    output[never_congested] = [119, 119, 119]  # Grey

    # Congested pixels Color by onset time
    congested_mask = ~np.isnan(relative_onset)

    if np.any(congested_mask):
        normalized = norm(relative_onset)
        colored = cmap(normalized)
        output[congested_mask] = (colored[congested_mask, :3] * 255).astype(np.uint8)
    return output

# In this experiement we calculate the time onset value for the avraged weekdays and days-off
Firste we calculate the average congestion level for road segment across dayweeks for the same time-slut


In [ ]:
def compute_onset(grouped_by_time):
    onset_map = None
    onset_detected = None
    road_mask = None

    for idx, time_key in enumerate(sorted(grouped_by_time.keys())):
        image_paths = grouped_by_time[time_key]

        wm = average_img_weight_map(image_paths)

        if onset_map is None:
            h, w = wm.shape
            onset_map = np.full((h, w), np.nan, dtype=np.float32)
            onset_detected = np.zeros((h, w), dtype=bool)
            road_mask = np.zeros((h, w), dtype=bool)

        congestion_mask = (wm < THETA) & (wm > 0)
        road_mask |= (wm > 0)

        new_onset_mask = congestion_mask & (~onset_detected)
        onset_map[new_onset_mask] = idx
        onset_detected[new_onset_mask] = True

        del wm

    return onset_map, road_mask

def plot_onset_map(relative_onset, road_mask,
                   grouped_by_time, title,
                   onset_congestion_map_dir, save_plot=True):

    t_min = np.nanmin(relative_onset)
    t_max = np.nanmax(relative_onset)
    time_labels = sorted(grouped_by_time.keys())
    custom_colors = ["#FF0000", "#00FF00", "#0000FF"]
    cmap = mcolors.LinearSegmentedColormap.from_list("Custom onset", custom_colors, N=256)
    norm_obj = plt.Normalize(vmin=t_min, vmax=t_max)
    onset_img = prepare_onset_map_plt(relative_onset, road_mask, cmap, norm_obj)
    fig, ax = plt.subplots(figsize=(8,6), dpi=1200)
    ax.imshow(onset_img)
    ax.axis("off")

    # Save Tight Image
    if save_plot:
        img_name = f"{title} - tight"
        img_path = os.path.join(onset_congestion_map_dir, img_name)
        plt.savefig(img_path, bbox_inches='tight', pad_inches=0)

    ax.set_title(title)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_obj)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    ticks = np.linspace(t_min, t_max, 24)
    ticks_labels = []

    for t in ticks:
        idx = int(round(t))
        idx = max(0, min(idx, len(time_labels) - 1))
        h, m = map(int, time_labels[idx].split(":"))
        ticks_labels.append(f"{h:02d}:{m:02d}")

    cbar.set_ticks(ticks)
    cbar.set_ticklabels(ticks_labels)
    cbar.set_label("Congestion Onset Time")

    # Save Image figure
    if save_plot:
        img_path = os.path.join(onset_congestion_map_dir, title)
        plt.savefig(img_path, bbox_inches='tight', pad_inches=0.05)

    plt.show()

for city_name, city_path_list in city_dict.items():
    print(f"Start Processing ({city_name})")
    grouped_weekday = defaultdict(list)
    grouped_friday = defaultdict(list)

    for city_path in tqdm(city_path_list, f"{city_name} grouping"):
        metadata = parse_filename(city_path)
        ts = metadata["timestamp"]

        time_key = ts.strftime("%H:%M")

        if ts.strftime("%a") == "Fri":
            grouped_friday[time_key].append(city_path)
        else:
            grouped_weekday[time_key].append(city_path)

    onset_weekday, road_weekday = compute_onset(grouped_weekday)
    onset_friday, road_friday = compute_onset(grouped_friday)

    data_paht = os.path.join(onset_congestion_map_dir, f'Congestion Onset ({city_name}).npz')
    np.savez(data_paht, onset_weekday=onset_weekday, road_weekday=road_weekday,
             grouped_weekday=grouped_weekday, onset_friday=onset_friday,
             road_friday=road_friday, grouped_friday=grouped_friday)
    print(city_path)
    plot_onset_map(
        onset_weekday,
        road_weekday,
        grouped_weekday,
        f"Congestion Onset Time Map Weekdays - {city_name}",
        onset_congestion_map_dir,
    )
    if onset_friday is not None:
        plot_onset_map(
            onset_friday,
            road_friday,
            grouped_friday,
            f"Congestion Onset Time Map Friday - {city_name}",
            onset_congestion_map_dir,
        )
    print(city_path)
    print("====")


## Create overly map using OpenStreatMap

In [ ]:
# Create overlay onset map
# Overlay image over a map
def latlon_to_tile(lat_deg: float, lon_deg: float, zoom: int):
    """
    Convert latitude/longitude to XYZ tile coordinates.
    """
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    x_tile = int((lon_deg + 180.0) / 360.0 * n)
    y_tile = int((1.0 - math.log(math.tan(lat_rad) + (1 / math.cos(lat_rad))) / math.pi) / 2.0 * n)
    return x_tile, y_tile

def tile_bounds(x: int, y: int, zoom: int, radius: int):
    """
    Return lat/lon bounds of the tile as (north, west, south, east)
    """
    n = 2.0 ** zoom
    lon_w = (x - radius) / n * 360.0 - 180.0
    lon_e = (x + radius + 1) / n * 360.0 - 180.0
    lat_n = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y - radius) / n))))
    lat_s = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + radius + 1) / n))))
    return lat_n, lon_w, lat_s, lon_e

def make_black_transparent(img_path):
    img = Image.open(img_path).convert("RGBA")
    arr = np.array(img)

    # Identify black pixels
    black_mask = np.all(arr[:, :, :3] == [0, 0, 0], axis=-1)

    # Set alpha channel
    arr[black_mask, 3] = 0      # transparent
    arr[~black_mask, 3] = 255   # fully visible

    return arr

overlay_maps_dict = {}
# Iterate over each city directory
image_path_list = glob.glob(os.path.join(onset_congestion_map_dir, "*.png"))
tight_maps_list = [map for map in image_path_list if "tight" in map]

if len(tight_maps_list) == 0:
    print("Found no tight maps to overlay")
else:
    for img_path in tqdm(tight_maps_list, "Generating Maps"):
        match_re = re.search(r"-\s*([^-]+?)(?:\s*-\s*tight)?\.[a-zA-Z0-9]+$",
                          img_path,
                          re.IGNORECASE)
        if match_re:
            city = match_re.group(1).strip()

        if city not in overlay_maps_dict:
            overlay_maps_dict[city] = []
        overlay_maps_dict[city].append(img_path)

        city_cords = CITY_CORD_DICT[city]
        zoom = 14
        radius = 6
        x, y = latlon_to_tile(city_cords['lat'], city_cords['lon'], zoom)
        north, west, south, east = tile_bounds(x, y, zoom, radius)

        map = folium.Map(location=[city_cords['lat'], city_cords['lon']], zoom_start=zoom)
        overlay_img = make_black_transparent(img_path)

        folium.raster_layers.ImageOverlay(
            name=f"Onset Congestion ({city})",
            image=overlay_img,
            bounds=[[south, west], [north, east]],
            opacity=0.55,
        ).add_to(map)

        folium.LayerControl().add_to(map)

        map.save(os.path.join(onset_overlay_map_dir,
                            f"onset_overlay_map - {city}.html"))

# Create Congestion Timelapse Video

In [ ]:
def extract_datetime(path):
    # Extract timestamp using regex
    match = re.search(r"\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}\.\d+\+\d{2}-\d{2}", path)

    if not match:
        raise ValueError(f"No datetime found in: {path}")

    dt_str = match.group()

    # Fix format:
    # Replace time '-' with ':' only in time part
    date_part, time_part = dt_str.split("T")

    # Split timezone
    time_main, tz = time_part.split("+")

    # Fix time
    time_main = time_main.replace("-", ":")

    # Fix timezone (+03-00 → +03:00)
    tz = tz.replace("-", ":")

    iso_str = f"{date_part}T{time_main}+{tz}"

    return datetime.fromisoformat(iso_str)

fps = 1
for city_name, city_path_list in city_dict.items():
    print(f"Start Creating Video for {city_name} City")

    # Read first frame to get size
    first_img = cv2.imread(city_path_list[0])
    height, width, _ = first_img.shape

    out = cv2.VideoWriter(
        os.path.join(traffic_timelapse_dir, f"TimeLapse-{city_name}.mp4"),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    images_list = sorted(city_path_list[::4], key=extract_datetime)
    for img_path in tqdm(images_list):
        img = cv2.imread(img_path)

        if img is None:
            print(f"Failed to load {img_path}")
            continue

        metadata = parse_filename(img_path)
        ts = metadata["timestamp"]

        text = f"{city_name} - {ts.strftime('%a %d/%m %H:%M')}"
        # Draw text
        cv2.putText(
            img,
            text,
            (20, img.shape[0] - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            8.0,
            (255, 255, 255),
            16,
            cv2.LINE_AA
        )

        # WRITE immediately (no list!)
        out.write(img)

        # Free memory explicitly
        del img

    out.release()
